# 02c — Stage 1 Winner Selection

Reads the standardized comparison outputs from `02a` (Time Series) and `02b` (ML), picks a winning model per variable, and stitches the corresponding forecast into one combined Stage 1 dataset for Stage B.

## Section 0 — Imports & Load

Both `winners_*.csv` files share an identical schema (verified in `02b`),
so they can be loaded and compared directly.

In [1]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go


LOCAL_OUTPUT = Path.cwd().parent / 'Stage1_Outputs'

ml_winners = pd.read_csv(LOCAL_OUTPUT / 'winners_ML_univariate.csv')
ts_winners = pd.read_csv(LOCAL_OUTPUT / 'winners_TS_univariate.csv')

print(f'ML winners: {ml_winners.shape}')
print(f'TS winners: {ts_winners.shape}')

ML winners: (12, 13)
TS winners: (12, 13)


## Section 1 — Winner Selection

Three-tier decision per variable:

1. **Stability gate** — if only one family's forecast is stable, it wins outright
2. **Validated gate** — if only one family significantly beat naive persistence
   (per its own DM test), it wins regardless of RMSE
3. **RMSE comparison** — reached only when both candidates are equally stable
   and equally validated (or equally unvalidated); lower `RMSE (primary)` wins

Winners that are unvalidated in both families are flagged explicitly — these
are the "less-bad of two" cases, not evidence either model actually beats
naive persistence.

In [2]:
combined = pd.concat([ml_winners, ts_winners], ignore_index=True)

selection_rows = []
for var in sorted(combined['Variable'].unique()):
    sub = combined[combined['Variable'] == var].set_index('Family')
    ml_row = sub.loc['ML_univariate']
    ts_row = sub.loc['TS_univariate']

    ml_stable = ml_row['Stable'] == 'Yes'
    ts_stable = ts_row['Stable'] == 'Yes'
    ml_valid = ml_row['Validated'] == 'Yes'
    ts_valid = ts_row['Validated'] == 'Yes'

    if ml_stable and not ts_stable:
        winner, reason = 'ML_univariate', 'only ML stable'
    elif ts_stable and not ml_stable:
        winner, reason = 'TS_univariate', 'only TS stable'
    elif ml_valid and not ts_valid:
        winner, reason = 'ML_univariate', 'only ML validated'
    elif ts_valid and not ml_valid:
        winner, reason = 'TS_univariate', 'only TS validated'
    elif ml_row['RMSE (primary)'] < ts_row['RMSE (primary)']:
        winner, reason = 'ML_univariate', 'lower RMSE'
    else:
        winner, reason = 'TS_univariate', 'lower RMSE'

    winner_row = ml_row if winner == 'ML_univariate' else ts_row

    selection_rows.append({
        'Variable': var,
        'Winner': winner,
        'Model': winner_row['Model'],
        'RMSE (primary)': winner_row['RMSE (primary)'],
        'Validated': winner_row['Validated'],
        'Reason': reason,
    })

selection = pd.DataFrame(selection_rows)
display(selection)

unvalidated = selection[selection['Validated'] == 'No']
if len(unvalidated) > 0:
    print(f'\nWARNING: {len(unvalidated)} of {len(selection)} winners are unvalidated')
    print('(neither family significantly beat naive persistence for these variables):')
    print(', '.join(unvalidated['Variable'].tolist()))

,Variable,Winner,Model,RMSE (primary),Validated,Reason
0,us_bond_yield_10y,ML_univariate,LASSO,1.5064,Yes,only ML validated
1,us_consumer_confidence,TS_univariate,AR,0.6148,No,lower RMSE
2,us_cpi,ML_univariate,KRR,2.7219,Yes,only ML validated
3,us_credit_qoq_growth,ML_univariate,XGBoost,0.6246,No,lower RMSE
4,us_gdp_yoy_growth,ML_univariate,XGBoost,0.4052,Yes,lower RMSE
5,us_house_price_yoy,ML_univariate,Ridge,1.8698,Yes,only ML validated
6,us_indprod_yoy,TS_univariate,SARMA,0.8817,No,lower RMSE
7,us_oil_yoy,ML_univariate,Elastic Net,23.5219,Yes,only ML validated
8,us_reer_diff,ML_univariate,KRR,2.7898,No,lower RMSE
9,us_sp500_log_ret,ML_univariate,Ridge,0.0756,No,lower RMSE



(neither family significantly beat naive persistence for these variables):
us_consumer_confidence, us_credit_qoq_growth, us_indprod_yoy, us_reer_diff, us_sp500_log_ret, us_vix_log_ret


## Section 2 — Load Regressor Files

**Two different historical series exist for the 12 target variables, and
they're not interchangeable.** `SARIMA_regressors_US_Q.csv` and
`ML_regressors_US_Q.csv` both carry COVID (2020 Q1-2021 Q4) as NaN, because
both source notebooks deliberately excluded that period *before fitting* -
a reasonable choice for training, since an extreme, non-representative
period shouldn't dominate model estimation.

But that's a modeling-time decision, not necessarily what the final
historical record should look like. Stage B has already committed to a
different philosophy for this same period: rather than excluding it, Stage
B applies a cubic-spline adjustment and keeps all ~137 quarters intact. If
this notebook hands Stage B a regressor set with a COVID-shaped hole in it,
Stage B either has to drop those rows anyway (silently undoing its own
spline-smoothing effort) or invent a fix here that arguably belongs
upstream.

So history here is rebuilt from the **complete** EDA-cleaned source
(`EDA_analytical_US_Q.csv`), before COVID exclusion, independently of what
`02a`/`02b` used for fitting. Only the forecast tail (2026 Q1 onward, where
COVID is irrelevant either way) still comes from each winning family's file.
Any COVID treatment for the regressors themselves - matching Stage B's
spline approach, leaving as raw actuals, or something else - is deliberately
left as a Stage B decision, not made here.

**Caveat:** the 8 derived variables (YoY/diff/log-diff transforms) are
re-derived here using the same transform definitions as `02a`'s
`DERIVED_VARS` - reconstructed for this notebook rather than imported
directly, so worth a direct diff against `02a`'s actual `apply_transform`/
`build_derived` functions before trusting this fully.

In [ ]:
FORECAST_START = pd.Timestamp('2026-03-31')
GITHUB_RAW = 'https://raw.githubusercontent.com/hogandan85/ST-498/refs/heads/main/Data%20Collection'

eda_complete = pd.read_csv(f'{GITHUB_RAW}/EDA_analytical_US_Q.csv', index_col=0, parse_dates=True)

DIRECT_LEVEL_VARS = ['us_unemployment', 'us_cpi', 'us_consumer_confidence', 'us_bond_yield_10y']
DERIVED_VARS = {
    'us_gdp_yoy_growth':    ('us_real_gdp',             'pct_change_yoy', 4),
    'us_house_price_yoy':   ('us_house_price_idx',      'pct_change_yoy', 4),
    'us_indprod_yoy':       ('us_industrial_production','pct_change_yoy', 4),
    'us_oil_yoy':           ('us_oil_price',            'pct_change_yoy', 4),
    'us_reer_diff':         ('us_reer',                 'diff',           1),
    'us_credit_qoq_growth': ('us_credit',               'pct_change_qoq', 1),
    'us_sp500_log_ret':     ('us_sp500_close',          'log_diff',       1),
    'us_vix_log_ret':       ('us_vix',                  'log_diff',       1),
}

def apply_transform(series, method, periods):
    if method in ('pct_change_yoy', 'pct_change_qoq'):
        return series.pct_change(periods) * 100
    elif method == 'diff':
        return series.diff(periods)
    elif method == 'log_diff':
        return np.log(series).diff(periods)
    raise ValueError(f'Unknown method: {method}')

history_complete = eda_complete.copy()
for var, (src, method, periods) in DERIVED_VARS.items():
    history_complete[var] = apply_transform(eda_complete[src], method, periods)

history_complete = history_complete.drop(columns=['delinquency_spline'], errors='ignore')

history_complete = history_complete.loc[history_complete.index < FORECAST_START]

target_cols = DIRECT_LEVEL_VARS + list(DERIVED_VARS.keys())
print(f'Total columns carried through: {history_complete.shape[1]} (12 target + {history_complete.shape[1] - 12} other EDA columns)')

sarima_full = pd.read_csv(LOCAL_OUTPUT / 'SARIMA_regressors_US_Q.csv', index_col=0, parse_dates=True)
ml_full = pd.read_csv(LOCAL_OUTPUT / 'ML_regressors_US_Q.csv', index_col=0, parse_dates=True)
sarima_forecast = sarima_full.loc[sarima_full.index >= FORECAST_START]
ml_forecast = ml_full.loc[ml_full.index >= FORECAST_START]

print(f'Complete history:  {history_complete.shape}  ({history_complete.index.min()} to {history_complete.index.max()})')
print(f'SARIMA forecast:   {sarima_forecast.shape}')
print(f'ML forecast:       {ml_forecast.shape}')

Total columns carried through: 42 (12 target + 30 other EDA columns)
Complete history:  (144, 42)  (1990-03-31 00:00:00 to 2025-12-31 00:00:00)
SARIMA forecast:   (20, 46)
ML forecast:       (20, 12)


## Section 3a — Stitch the Final Combined Dataset

For each of the 12 target variables, take its forecast column from whichever
family won in Section 1. History (all 46 SARIMA columns, since only 12 are
the actual targets - the rest are raw/intermediate series) is untouched;
only the 20 forecast rows differ by variable.

In [4]:
final_forecast = pd.DataFrame(index=sarima_forecast.index)

for _, row in selection.iterrows():
    var = row['Variable']
    if row['Winner'] == 'ML_univariate':
        final_forecast[var] = ml_forecast[var]
    else:
        final_forecast[var] = sarima_forecast[var]

final_combined = pd.concat([history_complete, final_forecast])

print(f'Final combined dataset: {final_combined.shape}')
print(f'Index: {final_combined.index.min()} to {final_combined.index.max()}')
display(final_forecast.tail(20))

Final combined dataset: (164, 42)
Index: 1990-03-31 00:00:00 to 2030-12-31 00:00:00


,us_bond_yield_10y,us_consumer_confidence,us_cpi,us_credit_qoq_growth,us_gdp_yoy_growth,us_house_price_yoy,us_indprod_yoy,us_oil_yoy,us_reer_diff,us_sp500_log_ret,us_unemployment,us_vix_log_ret
2026-03-31,4.117406,100.529139,2.871715,0.798847,2.183512,-1.294090,0.490196,-4.950448,0.063140,0.016405,4.427830,0.046093
2026-06-30,4.148768,100.511958,2.876021,0.866835,2.122606,-1.112064,0.193229,3.638101,0.111524,0.015989,4.498502,-0.021239
2026-09-30,4.176011,100.314399,2.773675,1.058027,2.129373,-0.793645,0.288558,10.991931,0.086912,0.018351,4.545147,0.024111
2026-12-31,4.241293,100.075863,2.961762,1.119772,1.947717,-0.595088,0.808003,17.565841,0.083960,0.020388,4.575475,0.001186
2027-03-31,4.274488,99.890183,2.942783,0.910654,2.331483,-0.394574,1.107166,16.174504,0.084992,0.015151,4.574252,0.014875
2027-06-30,4.311769,99.788016,2.973871,1.097557,2.493856,-0.271294,1.321493,16.036616,0.085156,0.016154,4.633514,0.007802
2027-09-30,4.346247,99.755683,3.071259,1.131141,2.641103,-0.142100,1.391400,11.376795,0.085115,0.018135,4.561535,0.011781
2027-12-31,4.377879,99.762485,3.048889,0.829095,2.727406,0.110142,1.399968,11.228218,0.085106,0.018441,4.558980,0.009668
2028-03-31,4.404749,99.780772,3.090732,1.277989,2.858373,0.378122,1.418426,11.149917,0.085108,0.018444,4.546688,0.010827
2028-06-30,4.435031,99.794520,3.120805,1.057321,3.029125,0.645074,1.426891,9.696476,0.085108,0.018316,4.530195,0.010203


## Section 3b — Back-Derive Remaining Forecast Columns

The 12 target variables are complete for 2026-2030, but two other column
types are still NaN for the forecast period: `covid_dummy` (no COVID in the
forecast horizon, so this is unambiguous) and the `_L{n}` lag-feature columns
plus `us_bond_yield_10y_d1`, all of which can be computed directly from the
already-forecasted target series.

**This never touches historical values** - every fill below uses `.fillna()`,
which only replaces genuinely empty cells. Whatever is already in the
historical portion of any lag column stays exactly as it was, including any
inconsistency with a simple shift (a separate, open question for Danny about
`02a`'s `LAG_SPEC`, not something resolved here).

In [5]:
final_combined['covid_dummy'] = final_combined['covid_dummy'].fillna(0)

final_combined['us_bond_yield_10y_d1'] = final_combined['us_bond_yield_10y_d1'].fillna(
    final_combined['us_bond_yield_10y'].diff(1))

import re
lag_pattern = re.compile(r'^(.+)_L(\d+)$')
lag_cols_filled = []
for col in final_combined.columns:
    m = lag_pattern.match(col)
    if m:
        base_var, lag_n = m.group(1), int(m.group(2))
        if base_var in final_combined.columns:
            shifted = final_combined[base_var].shift(lag_n)
            final_combined[col] = final_combined[col].fillna(shifted)
            lag_cols_filled.append(col)
        else:
            print(f'  WARNING: base variable "{base_var}" for "{col}" not found - left as-is')

print(f'Back-derived {len(lag_cols_filled)} lag columns for the forecast period:')
print(lag_cols_filled)

def reconstruct_level(full_index, base_history, forecast_transformed, method, periods):
    combined = base_history.reindex(full_index)
    idx_list = list(full_index)
    for date in forecast_transformed.index:
        pos = idx_list.index(date)
        base_val = combined.iloc[pos - periods]
        val = forecast_transformed.loc[date]
        if method in ('pct_change_yoy', 'pct_change_qoq'):
            combined.iloc[pos] = base_val * (1 + val / 100)
        elif method == 'diff':
            combined.iloc[pos] = base_val + val
        elif method == 'log_diff':
            combined.iloc[pos] = base_val * np.exp(val)
    return combined

reconstructed_levels = {}
for tgt, (src, method, periods) in DERIVED_VARS.items():
    if src not in reconstructed_levels:
        reconstructed_levels[src] = reconstruct_level(
            final_combined.index, final_combined[src], final_forecast[tgt], method, periods)
    final_combined[src] = final_combined[src].fillna(reconstructed_levels[src])

final_combined['us_reer_log_ret'] = final_combined['us_reer_log_ret'].fillna(
    np.log(reconstructed_levels['us_reer']).diff())
final_combined['us_oil_log_ret'] = final_combined['us_oil_log_ret'].fillna(
    np.log(reconstructed_levels['us_oil_price']).diff())
final_combined['us_credit_yoy_growth'] = final_combined['us_credit_yoy_growth'].fillna(
    reconstructed_levels['us_credit'].pct_change(4) * 100)

forecast_rows = final_combined.loc[final_combined.index >= FORECAST_START]
print(f'\nRemaining NaN in forecast period: {forecast_rows.isna().sum().sum()}')
remaining = forecast_rows.isna().sum()
print(remaining[remaining > 0])

Back-derived 13 lag columns for the forecast period:
['us_house_price_yoy_L3', 'us_unemployment_L0', 'us_consumer_confidence_L2', 'us_credit_qoq_growth_L6', 'us_gdp_yoy_growth_L2', 'us_cpi_L0', 'us_bond_yield_10y_d1_L2', 'us_sp500_log_ret_L4', 'us_indprod_yoy_L3', 'us_reer_diff_L0', 'us_oil_yoy_L2', 'us_vix_log_ret_L0', 'us_cpi_L6']

Remaining NaN in forecast period: 80
us_bond_yield_1y           20
us_term_spread             20
us_delinquency_rate        20
us_delinquency_rate_raw    20
dtype: int64


## Section 4 — History vs Forecast Visualization

One panel per variable, all 12 in a single figure. Solid line = history,
dashed line = forecast, shaded region marks the forecast horizon
(2026 Q1 onward). Panel titles show which family won each variable.

In [6]:
variables = selection['Variable'].tolist()
n_cols = 3
n_rows = -(-len(variables) // n_cols)

titles = [f"{row['Variable']}<br><sup>{row['Winner']} - {row['Model']}</sup>"
          for _, row in selection.iterrows()]

fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titles,
                     vertical_spacing=0.08, horizontal_spacing=0.06)

for idx, var in enumerate(variables):
    r, c = idx // n_cols + 1, idx % n_cols + 1
    hist = history_complete[var]
    fcst = final_forecast[var]

    fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode='lines',
                              line=dict(color='#1f77b4'), showlegend=False),
                  row=r, col=c)
    fig.add_trace(go.Scatter(x=fcst.index, y=fcst.values, mode='lines',
                              line=dict(color='#d62728', dash='dash'), showlegend=False),
                  row=r, col=c)
    fig.add_vrect(x0=FORECAST_START, x1=fcst.index.max(),
                  fillcolor='grey', opacity=0.1, line_width=0, row=r, col=c)

COVID_START = pd.Timestamp('2020-03-31')
COVID_END = pd.Timestamp('2021-12-31')

for idx, var in enumerate(variables):
    r, c = idx // n_cols + 1, idx % n_cols + 1
    fig.add_vrect(x0=COVID_START, x1=COVID_END,
                  fillcolor='#F4A460', opacity=0.15, line_width=0, row=r, col=c)

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=12, color='#F4A460', symbol='square'),
    name='COVID period (2020 Q1-2021 Q4)', showlegend=True))

fig.update_layout(
    height=280 * n_rows,
    title=dict(text='<b>History (solid) vs Forecast (dashed) - All 12 Variables</b>',
               x=0.5, xanchor='center', font=dict(size=18)),
    template='plotly_white', showlegend=True,
    legend=dict(orientation='h', y=1.055, x=0.5, xanchor='center'),
    margin=dict(t=100))
fig.show()

## Section 5 — Export

Two outputs: the final combined dataset (complete history + winning
forecast per variable) that Stage B will read directly, and the
winner-selection summary as an audit trail of which model won each
variable and why.

In [8]:
DATA_COLLECTION_DIR = Path.cwd().parent / 'Data Collection'

final_combined.to_csv(DATA_COLLECTION_DIR / 'Stage1_final_regressors_US_Q.csv')
selection.to_csv(LOCAL_OUTPUT / 'Stage1_winner_selection_summary.csv', index=False)

print(f"Saved: {DATA_COLLECTION_DIR / 'Stage1_final_regressors_US_Q.csv'}  (shape={final_combined.shape})")
print(f"Saved: {LOCAL_OUTPUT / 'Stage1_winner_selection_summary.csv'}  (shape={selection.shape})")

Saved: c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection\Stage1_final_regressors_US_Q.csv  (shape=(164, 42))
Saved: c:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Stage1_Outputs\Stage1_winner_selection_summary.csv  (shape=(12, 6))
